In [ ]:
from langchain_groq import ChatGroq

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b", api_key=GROQ_API_KEY)

In [ ]:
result=llm.invoke("Hey")

In [ ]:
print(result)

In [ ]:
print(result.response_metadata)

In [ ]:
print(result.content)

## Prompt Template 
it can be used for user prompt only

In [ ]:
from langchain_core.prompts import PromptTemplate

In [ ]:
PromptTemplate.from_template(template="Explain this topic {topic}",input=["topic"])

In [ ]:
input_temp = PromptTemplate.from_template("Explain this topic {topic}",input = ["topic"])

In [ ]:
input= input_temp.invoke({"topic": "BERT model"})

In [ ]:
llm.invoke(input).content

In [ ]:
data = PromptTemplate.from_template("Explain the {topic} into {no_points} ",input=["topic","no_points"])

In [ ]:
input_data = data.invoke({"topic":"Langchain","no_points":"5"})

In [ ]:
llm.invoke(input_data).content

### ChatPromptTemplate

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
chat_temp = ChatPromptTemplate.from_messages(
    [
        ("system","You are helpful Science teacher only explain science topics strictly."),
        ("user","explain this topic {topic}")
    ]
)

In [ ]:
chat_temp.messages[0]

In [ ]:
chat_temp.messages[1]

In [ ]:
input = chat_temp.invoke({"topic":"photosynthesis"})

In [ ]:
input

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model = "openai/gpt-oss-20b",api_key=os.environ["GROQ_API_KEY"])

In [ ]:
llm.invoke(input)

In [ ]:
chat_temp2 = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful maths teacher only explain maths topic strictly.only give response in JSON"),
        ("user","explain this topic {topic}")
    ]
)

In [ ]:
res = llm.invoke(chat_temp2.invoke({"topic":"integration"}))

In [ ]:
print(res.content)

In [ ]:
print(res.content)

### Parser

### String Output Parser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
str_parser = StrOutputParser()

In [ ]:
chat_temp = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful maths teacher only explain maths topic strictly"),
        ("user","Explain this topic {topic}")
    ]
)

In [ ]:
chat_temp

In [ ]:
response = llm.invoke(chat_temp.invoke({"topic":"intergration"}))

In [ ]:
response

In [ ]:
str_parser.invoke(response)

### Json Output Parser

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

In [ ]:
json_parser = JsonOutputParser()

In [ ]:
json_parser.get_format_instructions()

In [ ]:
chat_temp3 = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful maths teacher only explain maths topic strictly {JSON_FORMATTING}"),
        ("user","explain this topic {topic}")
    ]
).partial(JSON_FORMATTING = json_parser.get_format_instructions())

In [ ]:
chat_temp3

In [ ]:
input = chat_temp3.invoke({"topic":"differentiation"})

In [ ]:
response = llm.invoke(input)

In [ ]:
response

In [ ]:
final_output = json_parser.invoke(response)

In [ ]:
final_output

In [ ]:
final_output['topic']

In [ ]:
final_output['definition']

### Pydantic Output Parser

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

In [ ]:
from pydantic import BaseModel , Field

In [ ]:
class Student(BaseModel):
    name:str = Field(description="student name")
    roll_no: int = Field(description="student roll no")
    batch: str = Field(description="student academic batch")
    cgpa: float = Field(description="student cgpa")

In [ ]:
pydantic_parser = PydanticOutputParser(pydantic_object=Student)

In [ ]:
print(pydantic_parser.get_format_instructions())

In [ ]:
input_temp = ChatPromptTemplate.from_messages(
    [
        ("system","you are a helpful ai assistant.extract student information into this formation {PYDANTIC_FORMATTING}"),
        ("user","Student information is {student_info}")
]).partial(PYDANTIC_FORMATTING=pydantic_parser.get_format_instructions())

In [ ]:
input_temp

In [ ]:
input= input_temp.invoke({"student_info":"Student name is Krutika and Student roll no is 57 , batch is 2025 cgpa is 8.5"})

In [ ]:
input

In [ ]:
response = llm.invoke(input)

In [ ]:
response

In [ ]:
json_parser.invoke(response)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
system = SystemMessage(content="""You are a helpful ai assistant, extract student information""")
user = HumanMessage(content="Student name is Krutika and Student roll no is 57, batch is 2025 cgpa is 8.5")

In [ ]:
message = [system,user]

In [ ]:
message

In [ ]:
llm.invoke(message)

### Chains


In [ ]:

from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
import os

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b",api_key=os.environ["GROQ_API_KEY"])

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
input_temp = ChatPromptTemplate.from_messages([
    ("system","You are a experience post.write meaningful poem. do not response query other than poem"),
    ("user","Write a poem about {poem_topic}")
])

In [ ]:
input=input_temp.invoke({"poem_topic","rainfall"})

In [ ]:
res = llm.invoke(input)

In [ ]:
# input_temp2 = ChatPromptTemplate.from_messages([
#     ("system","You are an expert in summarizing. Summarize the poem in 5 lines"),
#     "user",input_temp
# ])

In [ ]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
out = StrOutputParser()

In [ ]:
chains = input_temp | llm | out 

In [ ]:
chains

In [ ]:
res = chains.invoke({"poem_topic","Summer Holidays"})

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

In [ ]:
json_parser = JsonOutputParser()

In [ ]:
json_parser.get_format_instructions()

#### Summarize poem part

In [ ]:
summarize_temp = ChatPromptTemplate.from_messages([
    ("system","You are an expert in summarizing. Summarize the poem in 5 points{JSON_FORMATING}"),
    ("user","Summarize this poem {poem}")
]).partial(JSON_FORMATING=json_parser.get_format_instructions())

In [ ]:
summarize_temp

In [ ]:
summarize_temp.invoke({"poem",res})

In [ ]:
step_2_input = summarize_temp.invoke({"poem",res}) # preparing input template

In [ ]:
res2 = llm.invoke(step_2_input)   # calling llm to summarize

In [ ]:
final_res = json_parser.invoke(res2)

In [ ]:
chains = input_temp | llm | out | summarize_temp | llm | json_parser

In [ ]:
chains.invoke({"poem_topic":"rainfall"})

## Tools

In [ ]:
def convert_marks_into_cgpa(marks):
    return round(marks/9.6)

In [ ]:
convert_marks_into_cgpa(90)

In [ ]:
from langchain_core.tools import tool

In [ ]:
@tool
def convert_marks_into_cgpa(marks:float) -> float:
    """This function converts marks into cgpa"""   #docstring
    return round(marks/9.6)

In [ ]:
convert_marks_into_cgpa.invoke({"marks":90})

In [ ]:
convert_marks_into_cgpa.__doc__

In [ ]:
@tool
def add(a:int,b:int)-> int:
    """This is add function"""
    return (a+b)

In [ ]:
add.invoke({"a":4,"b":4})

In [ ]:
add.__doc__

In [ ]:
add

In [ ]:
convert_marks_into_cgpa

In [ ]:
add.get_input_schema().model_json_schema()

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model = "openai/gpt-oss-20b",api_key=os.environ["GROQ_API_KEY"])

In [ ]:
llm.invoke("Hi")

In [ ]:
llm_with_tool = llm.bind_tools([add])

In [ ]:
response = llm_with_tool.invoke("What is the addition of 2 and 4")

In [ ]:
response

In [ ]:
response.tool_calls

In [ ]:
response.tool_calls[0]

In [ ]:
response.tool_calls[0]['args']

In [ ]:
tool_args = response.tool_calls[0]['args']

In [ ]:
add.invoke(tool_args)

In [ ]:
llm_with_tool.invoke("hi")

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

In [ ]:
message = []
while True:
    user_query = input("Enter user query")
    message.append(HumanMessage(user_query))
    if user_query.lower() == "exit":
        break

    ai_response = llm_with_tool.invoke(message)
    
    if ai_response.tool_calls:
        message.append(AIMessage(content=ai_response.content, tool_calls=ai_response.tool_calls))
        tool_args=ai_response.tool_calls[0]['args']
        tool_response=add.invoke(tool_args)
        message.append(ToolMessage(content=tool_response,tool_call_id=response.tool_calls[0]['id']))
        response=llm_with_tool.invoke(message)
        print(tool_response)
    else:
        message.append(AIMessage(ai_response.content))
        print(ai_response.content)

In [ ]:
message

### Structured Tool

In [ ]:
from langchain_core.tools import StructuredTool

#### this tool decorator only work for sync function or async function

In [ ]:
@tool
def addition(num1:float,num2:float)-> float:
    """This is addition 1"""
    return num1 + num2

In [ ]:
addition

In [ ]:
addition.invoke({"num1":3,"num2":4})

In [ ]:
from langchain_core.tools import StructuredTool

#### sync

In [ ]:
def addition1(num1:float,num2:float)-> float:
    """This function add num1 and num2"""
    return num1 + num2

#### Validation by pydantic

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class AdditionInput(BaseModel):
    num1:float=Field(gt=0,description="this field pass to function")
    num2:float=Field(gt=0,description="this field pass to function")

In [ ]:
addition_structured_tool=StructuredTool.from_function(
    name="addition",
    description="this function give addition of num1 and num2",
    func=addition1,
    args_schema=AdditionInput
)

In [ ]:
addition_structured_tool.invoke({"num1":4,"num2":5})

#### sync using tool decorator

In [ ]:
@tool(args_schema=AdditionInput,description="this functin add num1 and num2")
def addition(num1:float,num2:float)->float:
    """this function add num1 and num2"""
    return num1 + num2

In [ ]:
addition.invoke({"num1":4,"num2":5})

#### async tool

In [ ]:
import asyncio

In [ ]:
def addition1(num1:float,num2:float)->float:
    """this function add num1 and num2"""
    return num1 + num2

In [ ]:
async def addition2(num1:float,num2:float)->float:
    """this function add num1 and num2"""
    return num1 + num2

In [ ]:
addition_structured_tool=StructuredTool.from_function(
    name="addition",
    description="this is function to add",
    func=addition1,
    coroutine=addition2,
    args_schema=AdditionInput
)

In [ ]:
addition_structured_tool.invoke({"num1":3,"num2":5})

In [ ]:
await addition_structured_tool.ainvoke({"num1":3,"num2":5})

#### async using tool decorator

In [ ]:
@tool(args_schema=AdditionInput,description="this function to add")
async def addition2(num1:float,num2:float)->float:
    """this function add num1 and num2"""
    return num1 + num2

In [ ]:
await addition2.ainvoke({"num1":3,"num2":4})

### Base tool

In [ ]:
db_conn = "render.com"
db_pass = "mit_vpu"
db_user_name = "mit"
db_name = "mca"

In [ ]:
import sqlite3

In [ ]:
def get_student_name(roll_no:int):
    db_session=sqlite3.Connection()
    db_response=db_session.execute(f"SELECT name FROM student WHERE student.roll_no = {roll_no}")
    return db_response

In [ ]:
from langchain_core.tools import BaseTool

In [ ]:
class AdditionInput(BaseModel):
    num1:float=Field(gt=0,description="this field pass to function")
    num2:float=Field(gt=0,description="this field pass to function")

In [ ]:
from typing import Type

In [ ]:
class Addition(BaseTool):
    name:str="addition"
    description:str="this give addition to num1 and num2"
    args_schema:Type[BaseModel]=AdditionInput

    def _run(self,num1,num2):
        return num1 + num2

    async def _arun(self,num1, num2):
        return num1 + num2

In [ ]:
addition_base_tool=Addition()

In [ ]:
addition_base_tool.invoke({"num1":4,"num2":6})

In [ ]:
await addition_base_tool.ainvoke({"num1":4,"num2":6})